# SH17 YOLOv9s benchmark notebook

Notebook này dùng dataset ở `E:/data/SH17`, tải weights vào `E:/models/yolov9`, lưu checkpoint dưới `E:/models/sh17_yolov9s_runs`, và ghi leaderboard vào `D:/DS-AI/SH17/artifacts/sh17_yolov9s`.

In [1]:
from pathlib import Path

import pandas as pd

from scripts.sh17_yolov9c_pipeline import (
    append_result,
    build_dataset_manifest,
    build_oversampled_train_manifest,
    ensure_model_weights,
    expand_experiments,
    load_experiment_config,
    train_one_experiment,
)


In [2]:
CONFIG_PATH = Path("configs/sh17_yolov9s_experiments.yaml")
cfg = load_experiment_config(CONFIG_PATH)

dataset_manifest = build_dataset_manifest(
    cfg["paths"]["dataset_root"],
    Path(cfg["paths"]["artifact_root"]) / "dataset",
)
weights_path = ensure_model_weights(cfg["paths"]["weights_dir"], "yolov9s.pt")
experiments = expand_experiments(cfg, dataset_manifest)

print("weights:", weights_path)
print("train images:", dataset_manifest.train_count)
print("val images:", dataset_manifest.val_count)


weights: E:\models\yolov9\yolov9s.pt
train images: 6479
val images: 1620


In [3]:
if any(exp.get("use_oversampled_train_manifest") for exp in experiments):
    oversampled_manifest = build_oversampled_train_manifest(
        train_manifest=dataset_manifest.train_manifest,
        labels_dir=Path(cfg["paths"]["dataset_root"]) / "labels",
        output_path=Path(cfg["paths"]["artifact_root"]) / "dataset" / "train_oversampled.txt",
        minority_class_ids={2, 4, 6, 10, 13, 16},
        repeat_factor=3,
    )
    for exp in experiments:
        if exp.get("use_oversampled_train_manifest"):
            exp["train_manifest_override"] = str(oversampled_manifest)

[(exp["name"], exp.get("train_manifest_override")) for exp in experiments]


[('yolov9s_baseline_640', None),
 ('yolov9s_tuned_640', None),
 ('yolov9s_multiscale_960', None),
 ('yolov9s_oversample_minority_960',
  'D:\\DS-AI\\SH17\\artifacts\\sh17_yolov9s\\dataset\\train_oversampled.txt')]

## Run order

Khuyến nghị chạy theo thứ tự sau để so sánh với baseline và các cải tiến trong paper summary:

1. `yolov9s_baseline_640`
2. `yolov9s_tuned_640`
3. `yolov9s_multiscale_960`
4. `yolov9s_oversample_minority_960`


In [ ]:
ACTIVE_EXPERIMENTS = [
    # "yolov9s_baseline_640",
    "yolov9s_tuned_640",
    "yolov9s_multiscale_960",
    "yolov9s_oversample_minority_960",
]

records = []
for exp in experiments:
    if exp["name"] not in ACTIVE_EXPERIMENTS:
        continue
    record = train_one_experiment(exp, cfg["paths"]["runs_root"])
    append_result(record, cfg["paths"]["artifact_root"])
    records.append(record)

pd.DataFrame(records).sort_values(["map50", "map"], ascending=False)


Ultralytics 8.4.57  Python-3.13.13 torch-2.12.0+cu126 CUDA:0 (NVIDIA GeForce RTX 4060 Ti, 16380MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=-1, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=D:\DS-AI\SH17\artifacts\sh17_yolov9s\dataset\sh17.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=200, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=E:\models\yolov9\yolov9s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolov9s_tuned_640, nbs=64, nms=False, opset=No

In [ ]:
leaderboard = pd.read_csv(Path(cfg["paths"]["artifact_root"]) / "leaderboard.csv")
leaderboard.sort_values(["map50", "map"], ascending=False).head(10)
